# Week12 Capstone project - PCA guided refinement

## Strategy
For Week 12, the search becomes more structured by using a **PCA-guided refinement** step.

The motivation is that with enough historical queries, some directions in the search space explain more variation than others. Instead of perturbing every coordinate equally, this notebook:

1. takes the strongest recent weekly points for each function
2. computes their centroid as a stable promising region
3. applies **PCA** to those top points to estimate the dominant direction of variation
4. generates a small, human-clean candidate set by moving mainly along that principal direction
5. ranks candidates with a local GP score that favours high mean, modest uncertainty and closeness to the strong region

This reduces randomness while preserving a little exploration along the directions that appear to matter most.


In [4]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.exceptions import ConvergenceWarning


In [5]:
data_dir = Path('data')
history = pd.read_csv(data_dir / 'weekly_results/Week12.csv')
history.tail(8)


,week,function,d,x1,x2,x3,x4,x5,x6,x7,x8,y
72,10,1,2,0.632500,0.627500,NaN,NaN,NaN,NaN,NaN,NaN,1.921027
73,10,2,2,0.507500,0.692500,NaN,NaN,NaN,NaN,NaN,NaN,0.663513
74,10,3,3,0.372500,0.572500,0.467500,NaN,NaN,NaN,NaN,NaN,-0.000179
75,10,4,4,0.397200,0.392800,0.360200,0.440800,NaN,NaN,NaN,NaN,0.202835
76,10,5,4,0.999999,0.000001,0.999999,0.999999,NaN,NaN,NaN,NaN,4440.480874
77,10,6,5,0.532000,0.178000,0.628000,0.882000,0.0280,NaN,NaN,NaN,-0.514559
78,10,7,6,0.015600,0.183200,0.289200,0.173800,0.3978,0.666,NaN,NaN,2.107605
79,10,8,8,0.035200,0.438600,0.012200,0.324200,0.9910,0.046,0.0982,0.7038,9.590175


In [9]:
def load_initial(function_id):
    X0 = np.load(data_dir / f'initial_data/F{function_id}_initial_inputs.npy')
    y0 = np.load(data_dir / f'initial_data/F{function_id}_initial_outputs.npy')
    return X0, y0

def load_combined(function_id, history):
    X0, y0 = load_initial(function_id)
    rows = history[history['function'] == function_id].sort_values('week')
    d = int(rows['d'].iloc[0])
    Xw = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)
    yw = rows['y'].to_numpy(dtype=float)
    X = np.vstack([X0, Xw])
    y = np.concatenate([y0, yw])
    return X, y, Xw, yw, d

def round_grid(x, grid):
    return np.clip(np.round(x / grid) * grid, 0.0, 1.0)

def format_point(x):
    return '-'.join(f'{float(v):.6f}' for v in x)


In [10]:
cfg = {
    1: {'grid': 0.0005,  'step': 0.0010},
    2: {'grid': 0.0025,  'step': 0.0050},
    3: {'grid': 0.0005,  'step': 0.0010},
    4: {'grid': 0.0002,  'step': 0.0004},
    5: {'grid': 0.000001, 'step': 0.000001},
    6: {'grid': 0.0010,  'step': 0.0020},
    7: {'grid': 0.0002,  'step': 0.0004},
    8: {'grid': 0.0001,  'step': 0.0002},
}

def beta_by_dimension(d):
    return 0.08 + 0.03 * d

def pca_direction(Xw):
    if Xw.shape[1] == 1:
        return np.ones(1)
    pca = PCA(n_components=1)
    pca.fit(Xw)
    vec = pca.components_[0]
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

def build_gp(seed, d):
    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * RBF(length_scale=np.ones(d), length_scale_bounds=(0.01, 1.5))
        + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
    )
    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=seed,
        n_restarts_optimizer=1,
    )

def build_pca_candidates(best_x, latest_x, centroid, direction, d, step, grid):
    cands = []
    anchors = [best_x, latest_x, centroid, (best_x + latest_x) / 2.0, (best_x + centroid) / 2.0, (2 * best_x + centroid) / 3.0]
    for a in anchors:
        cands.append(round_grid(a, grid))
    for mult in [-2, -1, -0.5, 0.5, 1, 2]:
        cands.append(round_grid(centroid + mult * step * direction, grid))
        cands.append(round_grid(best_x + mult * step * direction, grid))
    var_order = np.argsort(np.var(np.vstack([best_x, latest_x, centroid]), axis=0))[::-1]
    for i in var_order[:min(3, d)]:
        for s in (-1.0, 1.0):
            x = centroid.copy()
            x[i] += s * step
            cands.append(round_grid(x, grid))
    return np.unique(np.array(cands), axis=0)


In [ ]:
results = []
for function_id in range(1, 9):
    X, y, Xw, yw, d = load_combined(function_id, history)
    best_idx = int(np.argmax(yw))
    latest_x = Xw[-1].copy()
    best_x = Xw[best_idx].copy()
    top_idx = np.argsort(yw)[-min(5, len(yw)):]
    centroid = Xw[top_idx].mean(axis=0)
    direction = pca_direction(Xw[top_idx])
    grid = cfg[function_id]['grid']
    step = cfg[function_id]['step']
    beta = beta_by_dimension(d)
    cands = build_pca_candidates(best_x, latest_x, centroid, direction, d, step, grid)
    dist_hist = np.sqrt(((cands[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)) / np.sqrt(d)
    min_dist = dist_hist.min(axis=1)
    novelty_floor = grid / 2 if grid < 0.001 else 0.0005
    keep = min_dist >= novelty_floor
    cands = cands[keep]
    min_dist = min_dist[keep]
    gp = build_gp(1300 + function_id, d)
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        gp.fit(X, y)
    mean, std = gp.predict(cands, return_std=True)
    centroid_dist = np.sqrt(((cands - centroid) ** 2).sum(axis=1)) / np.sqrt(d)
    best_dist = np.sqrt(((cands - best_x) ** 2).sum(axis=1)) / np.sqrt(d)
    latest_dist = np.sqrt(((cands - latest_x) ** 2).sum(axis=1)) / np.sqrt(d)
    score = mean + beta * std - 0.12 * centroid_dist - 0.08 * best_dist - 0.03 * latest_dist + 0.02 * min_dist
    idx = int(np.argmax(score))
    chosen = cands[idx]
    print(f'Function {function_id}: {format_point(chosen)}')
    print(f'  centroid: {format_point(centroid)}')
    print(f'  best:     {format_point(best_x)}')
    print(f'  latest:   {format_point(latest_x)}')
    print()
    row = {'function': function_id, 'd': d}
    for i, v in enumerate(chosen, start=1):
        row[f'x{i}'] = round(float(v), 6)
    results.append(row)
week12_df = pd.DataFrame(results)
week12_df.to_csv('Week12_suggestions.csv', index=False)
week12_df
